In [11]:
import pandas as pd, sqlite3

df = pd.read_excel("week12_transactions.xlsx")
df.to_csv("week12_transactions.csv", index=False)   # Part 3 needs the CSV

conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)

print(df.shape)
df.head()

(2000, 7)


,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,46327,8,Gasabo,send,1000,0
1,T00002,46335,12,Nyarugenge,send,25000,0
2,T00003,46345,8,Muhanga,send,51000,0
3,T00004,46327,17,Huye,send,79000,0
4,T00005,46341,12,Huye,send,109000,0


In [2]:
q = """
SELECT txn_id, district, amount_rwf
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC, txn_id
LIMIT 5
"""
pd.read_sql(q, conn)

,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


In [3]:
q = "SELECT COUNT(*) AS flagged_count FROM transactions WHERE flagged = 1"
pd.read_sql(q, conn)

,flagged_count
0,11


In [4]:
q = """
SELECT txn_type,
       COUNT(*)        AS n,
       SUM(amount_rwf) AS total_rwf,
       AVG(amount_rwf) AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
"""
pd.read_sql(q, conn)

,txn_type,n,total_rwf,avg_rwf
0,send,868,62049000,71485.023041
1,paybill,330,29970000,90818.181818
2,receive,489,27024000,55263.803681
3,airtime,313,833000,2661.341853


In [5]:
q = """
SELECT district, SUM(amount_rwf) AS total_rwf
FROM transactions
GROUP BY district
ORDER BY total_rwf DESC
"""
pd.read_sql(q, conn)

,district,total_rwf
0,Muhanga,21076000
1,Gasabo,20738000
2,Huye,18736000
3,Nyarugenge,18091000
4,Kicukiro,11338000
5,Musanze,10490500
6,Rubavu,10110500
7,Nyagatare,9296000


In [6]:
q = """
SELECT hour, COUNT(*) AS n
FROM transactions
GROUP BY hour
ORDER BY n DESC
"""
pd.read_sql(q, conn)

,hour,n
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169
5,10,168
6,8,168
7,19,158
8,9,144
9,16,38


In [7]:
hour_counts = df["hour"].value_counts()

print(hour_counts.head())
print("Busiest hour:", hour_counts.idxmax(), "with", hour_counts.max(), "transactions")

# Confirm both agree
sql_result = pd.read_sql(q, conn)
print("Agree?", hour_counts.idxmax() == sql_result.loc[0, "hour"]
              and hour_counts.max() == sql_result.loc[0, "n"])

hour
20    185
17    183
18    176
12    170
11    169
Name: count, dtype: int64
Busiest hour: 20 with 185 transactions
Agree? True


In [8]:
import json

raw = '{"city": "Kigali", "temps": [21.5, 24.0, 26.2]}'   # from slide 10

data = json.loads(raw)
print(type(data))                                          # dict

print("City:", data["city"])                               # (a)
print("Number of readings:", len(data["temps"]))           # (b)
print("Average:", round(sum(data["temps"]) / len(data["temps"]), 2))   # (c)

<class 'dict'>
City: Kigali
Number of readings: 3
Average: 23.9


In [9]:
import requests, json

url = ("https://api.open-meteo.com/v1/forecast"
       "?latitude=-1.95&longitude=30.06"
       "&hourly=temperature_2m&forecast_days=1")

try:
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    data = r.json()
    print("Live API OK, status:", r.status_code)
except Exception as e:
    print("API failed:", e)
    with open("week12_api_sample.json") as f:      # offline fallback
        data = json.load(f)
    print("Using offline sample file")

Live API OK, status: 200


In [10]:
temps = data["hourly"]["temperature_2m"]
times = data["hourly"]["time"]

print("Hours:", len(temps))                                   # (a)
print("Max:", max(temps), "°C | Min:", min(temps), "°C")      # (b)

hottest_time = times[temps.index(max(temps))]
print("Hottest hour:", hottest_time)                          # (c)

Hours: 24
Max: 24.5 °C | Min: 16.0 °C
Hottest hour: 2026-11-22T13:00


In [12]:
import pandas as pd

weather = pd.DataFrame({"time": times, "temp_c": temps})
print(weather.head())

# (a) describe the temperature
print(weather["temp_c"].describe())

# (b) count hours >= 25 °C using a mask
mask = weather["temp_c"] >= 25
print("Hours >= 25 °C:", mask.sum())

               time  temp_c
0  2026-11-22T00:00    16.8
1  2026-11-22T01:00    16.3
2  2026-11-22T02:00    16.6
3  2026-11-22T03:00    16.6
4  2026-11-22T04:00    16.7
count    24.000000
mean     19.650000
std       2.892719
min      16.000000
25%      16.775000
50%      19.250000
75%      21.750000
max      24.500000
Name: temp_c, dtype: float64
Hours >= 25 °C: 0


In [16]:
import os
os.environ["JAVA_HOME"] = r"D:\DevTools\Java\jdk-21"

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("momo")
         .master("local[*]")
         .getOrCreate())

print("Spark version:", spark.version)

D:\Anaconda\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.3.0


In [17]:
sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)

print("Rows:", sdf.count())     # (a) row count
sdf.printSchema()               # (b) schema
sdf.show(5)                     # (c) first 5 rows

Rows: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+-----+----+----------+--------+----------+-------+
|txn_id| date|hour|  district|txn_type|amount_rwf|flagged|
+------+-----+----+----------+--------+----------+-------+
|T00001|46327|   8|    Gasabo|    send|      1000|      0|
|T00002|46335|  12|Nyarugenge|    send|     25000|      0|
|T00003|46345|   8|   Muhanga|    send|     51000|      0|
|T00004|46327|  17|      Huye|    send|     79000|      0|
|T00005|46341|  12|      Huye|    send|    109000|      0|
+------+-----+----+----------+--------+----------+-------+
only showing top 5 rows


In [18]:
# (a) transactions above 100,000 RWF
big = sdf.filter(sdf.amount_rwf > 100000)
print("Above 100k:", big.count())

# (b) show 5 flagged transactions
sdf.filter(sdf.flagged == 1).show(5)

Above 100k: 451
+------+-----+----+--------+--------+----------+-------+
|txn_id| date|hour|district|txn_type|amount_rwf|flagged|
+------+-----+----+--------+--------+----------+-------+
|T00279|46355|  19| Muhanga|    send|    127000|      1|
|T00326|46333|  22|    Huye|    send|    127000|      1|
|T00536|46353|  17|  Gasabo|    send|    127000|      1|
|T00655|46356|  17|  Rubavu|    send|    134000|      1|
|T00697|46327|   9| Muhanga|    send|    127000|      1|
+------+-----+----+--------+--------+----------+-------+
only showing top 5 rows


In [19]:
# (a) total amount per txn_type
sdf.groupBy("txn_type").sum("amount_rwf").show()

# (b) per-district totals, converted to Pandas and sorted
district_totals = sdf.groupBy("district").sum("amount_rwf").toPandas()
district_totals = district_totals.sort_values("sum(amount_rwf)", ascending=False)
print(district_totals)

# (c) Do the Spark answers match my SQL answers? YES:
# send = 62,049,000 and Muhanga leads with 21,076,000 in both SQL and Spark.

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+



D:\Anaconda\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


     district  sum(amount_rwf)
4     Muhanga         21076000
5      Gasabo         20738000
6        Huye         18736000
1  Nyarugenge         18091000
0    Kicukiro         11338000
2     Musanze         10490500
7      Rubavu         10110500
3   Nyagatare          9296000


In [20]:
sdf.createOrReplaceTempView("transactions")

q = """
SELECT txn_type,
       COUNT(*)        AS n,
       SUM(amount_rwf) AS total_rwf,
       AVG(amount_rwf) AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
"""
spark.sql(q).show()

# Why is it powerful that the same SQL runs on SQLite and on a cluster?
# Because analysts write the query once and it works at any scale: the same
# skill and the same code move from a laptop database to thousands of
# machines, so nothing needs to be rewritten when the data grows.

+--------+---+---------+------------------+
|txn_type|  n|total_rwf|           avg_rwf|
+--------+---+---------+------------------+
|    send|868| 62049000| 71485.02304147465|
| paybill|330| 29970000| 90818.18181818182|
| receive|489| 27024000|  55263.8036809816|
| airtime|313|   833000|2661.3418530351437|
+--------+---+---------+------------------+



In [21]:
## Part 4 · Choose Your Weapon

**1. Ministry of Health, PostgreSQL, monthly totals per district → SQL**
The data already lives in a database, so I would query it directly with SQL (GROUP BY) instead of copying it out first.

**2. 40,000-row CSV needing cleaning, three charts and a regression model → Pandas**
40,000 rows fit easily in my laptop's memory, and Pandas handles cleaning and plugs straight into charts and Scikit-learn.

**3. Telecom call records, 40 TB already on their cluster → Spark**
40 TB is far too big for one machine, so I would use Spark to run the work on the cluster where the data already sits.

**4. Dashboard that always shows today's weather without anyone uploading a file → API**
An API gives fresh data every time my code calls it, so the dashboard updates itself with no manual upload.

SyntaxError: invalid character '→' (U+2192) (1262294114.py, line 3)